# NFF-MTL on Steam — final runs and analysis (v0.3.1, clean)

This notebook reads the recorded final runs and recomputes summary analyses. With the default switches, "Run all" does not train a model, evaluate the test year, rerun comparisons, rename folders, or launch the optional hours audit. Setup installs the package in the Colab runtime, and the final cell writes summary CSV files to `paper_tables/`. Every reported model result comes from the run folders listed in the ledger (section 4).

The full lab record with all original cells and outputs is the archive copy
`NFF_MTL_Corrected_Colab_v0.3.1_archive.ipynb`; keep it read-only. New experiments, if any, must not be run from this notebook (see section 8).

Times in folder names and logs are UTC (Türkiye = UTC+3).

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd

def run(*args):
    """Run a command, stream its output, raise CalledProcessError on a non-zero exit; return RUN_DIR if printed."""
    proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    run_dir = None
    for line in proc.stdout:
        print(line, end='')
        if line.startswith('RUN_DIR='):
            run_dir = line.strip().split('=', 1)[1]
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, args)
    return run_dir

def cli(*args):
    """Run `python -m steam_nffmtl.cli` (used here only for `compare`)."""
    return run(sys.executable, '-m', 'steam_nffmtl.cli', *args)

## 1. Setup

A Colab CPU or GPU runtime works with the default switches. The package is unzipped only if it is not already present.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/steamRS')
REPO = Path('/content/steam-nffmtl-v0.3.1')
if not REPO.exists():
    run('unzip', '-q', '-o', DATA_DIR / 'steam-nffmtl-v0.3.1.zip', '-d', '/content/')
%cd {REPO}
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt')
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]')
sys.path.insert(0, str(REPO / 'src'))

In [ ]:
RF = DATA_DIR / 'runs_final'
FINAL_CFG = DATA_DIR / 'runs_tuning' / '2026-11-09-102238-tuning-tune' / 'final_selected.yaml'
FINAL_SET = ['--set', 'train.max_epochs=10']   # every final command used this override (decided on validation data)
COMPARE = {'test-positive': RF / '2026-11-18-144556-final-compare',
           'test-interaction': RF / '2026-11-18-145125-final-compare'}

RUN_CHECKS = False      # preflight + test suite (~1 minute)
RERUN_COMPARE = False   # recompute both comparisons from the ledger (~6 minutes each, CPU only)
MARK_ORPHANS = False    # rename unfinished, unused training folders to *-INTERRUPTED (section 5)
RERUN_AUDIT = False     # re-run check_hours.py even if hours_report.json exists

assert FINAL_CFG.exists(), FINAL_CFG
assert all(d.exists() for d in COMPARE.values()), COMPARE

## 2. Optional checks

In [ ]:
if RUN_CHECKS:
    cli('preflight', '--out', DATA_DIR / 'preflight')
    run(sys.executable, '-m', 'pytest', '-q', '-rs')
else:
    print('Checks skipped (RUN_CHECKS = False).')

## 3. Experiment record (what was run — do not re-run here)

| Stage | Date (UTC) | Command | Folder(s) |
| --- | --- | --- | --- |
| Prepare | 2026-11-09 | `prepare --config configs/default.yaml` | `runs_tuning/2026-11-09-093332-tuning-prepare` |
| Probe | 2026-11-09 | `train --config configs/probe.yaml --variant full --seeds 0`; `baselines --config configs/probe.yaml --split val --seed 0` | `runs_probe/2026-11-09-093942-pilot-train-full-seed0`, `runs_probe/2026-11-09-094046-pilot-baselines-val-seed0` |
| Pilot | 2026-11-09 | `train --config configs/pilot.yaml --variant full --seeds 0` | `runs_pilot/2026-11-09-094721-pilot-train-full-seed0` |
| Tuning | 2026-11-09 | `tune --config configs/default.yaml --models tagknn itemknn bpr als nffmtl --set tuning.n_trials=3 train.max_epochs=5` | `runs_tuning/2026-11-09-102238-tuning-tune` (`final_selected.yaml`) |
| 10-epoch check (validation only) | 2026-11-09 | `train --config FINAL_CFG --variant full --seeds 0 --set experiment.stage=tuning train.max_epochs=10` | `runs_final/2026-11-09-124130-tuning-train-full-seed0` |
| Final NFF-MTL: 3 variants × seeds 0–4 | 2026-11-10 → 2026-11-17 | `train --config FINAL_CFG --variant V --seeds S --set train.max_epochs=10`, then `evaluate --config FINAL_CFG --split test --run <train folder> --set train.max_epochs=10` | ledger (section 4) |
| Final baselines | 2026-11-10, 2026-11-18 | seed 0: all models; seeds 1–4: `--only Random BPR-MF ALS` | ledger |
| EASE tuning | 2026-11-18 | `tune --config configs/default.yaml --models ease --set baselines.ease.enabled=true tuning.n_trials=3` → `l2 = 100` | `runs_tuning/2026-11-18-103721-tuning-tune` |
| EASE test | 2026-11-18 | `baselines --config FINAL_CFG --split test --seed 0 --only EASE --set baselines.ease.enabled=true baselines.ease.l2=100.0 baselines.ease.device=cpu train.max_epochs=10` | `runs_final/2026-11-18-142502-final-baselines-test-seed0` |
| Comparisons | 2026-11-18 | `compare --config FINAL_CFG --reference NFF-MTL --split test-positive` (and `test-interaction`) | `runs_final/2026-11-18-144556-final-compare`, `runs_final/2026-11-18-145125-final-compare` |

**Deviations and incidents to report**

- Hardware: Colab L4 (not A100); Python 3.13, torch 2.14, numpy 2.4, pandas 2.2 (exact versions in each run's `env.json`). The test suite passed 63/63 on this runtime.
- Tuning used 3 candidates per model (equal budget). NFF-MTL trials were capped at 5 epochs; the cap was raised to 10 after one extra validation-only run, so NFF-MTL received one validation run more than the other models.
- EASE was added after the other models' test results were seen. It was tuned on validation only, with the same 3-trial budget. On the L4 the float64 inversion did not finish within an hour, so the test run used the CPU inversion (same computation).
- Interrupting a notebook cell does not stop the CLI process: the seed-2 training of 2026-11-12 finished in the background and is the one used. Unused attempts are renamed `*-INTERRUPTED`: EASE `2026-11-18-112503`, `2026-11-18-122839`; full seed 2 `2026-11-12-144140`. The partial full-seed-2 run `2026-11-14-151106` (accidental re-run, stopped after 3 epochs) is unused and is marked in section 5.

## 4. Final run ledger

The single source of truth: 15 NFF-MTL evaluation folders (3 variants × 5 seeds) and 6 baseline folders
(seed 0 with all models, seeds 1–4 with Random/BPR-MF/ALS, and EASE).

In [ ]:
LEDGER = {
    'baselines': ['2026-11-10-114002-final-baselines-test-seed0', '2026-11-18-085122-final-baselines-test-seed1',
                  '2026-11-18-091445-final-baselines-test-seed2', '2026-11-18-093726-final-baselines-test-seed3',
                  '2026-11-18-100018-final-baselines-test-seed4', '2026-11-18-142502-final-baselines-test-seed0'],
    'full': ['2026-11-10-112426-final-evaluate-test-full-seed0', '2026-11-12-110556-final-evaluate-test-full-seed1',
             '2026-11-12-144748-final-evaluate-test-full-seed2', '2026-11-13-121946-final-evaluate-test-full-seed3',
             '2026-11-13-135138-final-evaluate-test-full-seed4'],
    'no_content': ['2026-11-13-190729-final-evaluate-test-no_content-seed0', '2026-11-15-103653-final-evaluate-test-no_content-seed1',
                   '2026-11-15-131526-final-evaluate-test-no_content-seed2', '2026-11-16-094010-final-evaluate-test-no_content-seed3',
                   '2026-11-16-124644-final-evaluate-test-no_content-seed4'],
    'single_task': ['2026-11-17-110731-final-evaluate-test-single_task-seed0', '2026-11-17-191017-final-evaluate-test-single_task-seed1',
                    '2026-11-17-202246-final-evaluate-test-single_task-seed2', '2026-11-17-215014-final-evaluate-test-single_task-seed3',
                    '2026-11-17-231725-final-evaluate-test-single_task-seed4'],
}
result_dirs = [RF / name for names in LEDGER.values() for name in names]
missing = [p.name for p in result_dirs if not list((p / 'results').glob('*__test-positive.json'))]
assert not missing, f'Missing or incomplete result folders: {missing}'
rows = []
for p in result_dirs:
    m = json.loads((p / 'manifest.json').read_text())
    models = sorted({json.loads(j.read_text())['model'] for j in (p / 'results').glob('*__test-positive.json')})
    rows.append({'folder': p.name, 'command': m['command'], 'variant': m.get('variant', '-'), 'seed': m.get('seed'),
                 'models': ', '.join(models), 'train_folder': Path(m['checkpoint']).parent.name if 'checkpoint' in m else ''})
ledger = pd.DataFrame(rows)
print(len(ledger), 'result folders (expected 21)')
display(ledger)

## 5. Drive hygiene

Lists training folders that no evaluation uses. Unfinished ones are renamed `*-INTERRUPTED` only when
`MARK_ORPHANS = True`, and only while no CLI process is running.

In [ ]:
used = set(ledger['train_folder']) - {''}
orphans = [p for p in sorted(RF.glob('*-final-train-*')) if 'INTERRUPTED' not in p.name and p.name not in used]
for p in orphans:
    print('unused training folder:', p.name, '| finished:', (p / 'training.json').exists())
print('already marked:', [p.name for p in sorted(RF.glob('*INTERRUPTED*'))])
if MARK_ORPHANS:
    busy = subprocess.run(['pgrep', '-f', 'steam_nffmtl.cli'], capture_output=True, text=True).stdout.strip()
    assert not busy, 'A steam_nffmtl CLI process is running; wait until it finishes.'
    for p in orphans:
        if not (p / 'training.json').exists():
            p.rename(p.with_name(p.name + '-INTERRUPTED'))
            print('marked:', p.name)

## 6. Comparison tables and statistics

In [ ]:
if RERUN_COMPARE:
    for split in COMPARE:
        COMPARE[split] = Path(cli('compare', '--config', FINAL_CFG, '--reference', 'NFF-MTL', '--split', split,
                                  '--results', *result_dirs, *FINAL_SET))
COLS = ['runs', 'ndcg@10_mean', 'ndcg@20_mean', 'ndcg@20_sd', 'recall@10_mean', 'recall@20_mean', 'recall@20_sd',
        'precision@10_mean', 'hr@20_mean', 'coverage@20_mean', 'long_tail_share@20_mean', 'arp@20_mean',
        'auc_review_label_mean']
tables = {s: pd.read_csv(d / 'comparison_table.csv', index_col=0).sort_values('ndcg@20_mean', ascending=False)
          for s, d in COMPARE.items()}
for s, t in tables.items():
    print(s, '-', COMPARE[s].name)
    display(t[[c for c in COLS if c in t.columns]].round(4))

In [ ]:
for s, d in COMPARE.items():
    st = json.loads((d / 'comparison_stats.json').read_text())
    for metric in ['ndcg@20', 'recall@20']:
        holm = st['holm_seed_level'].get(metric, {})
        print(f'\n{s} | {metric} | seed-level paired t-test, Holm over {len(holm)} comparisons (difference = NFF-MTL - other)')
        for key, v in st['seed_level'].items():
            if key.endswith('| ' + metric):
                other = key.split(' vs ')[1].split(' |')[0]
                lo, hi = v['ci95_seed_t']
                print(f'  {other:22s} diff {v["mean_diff"]:+.5f}  95% CI [{lo:+.5f}, {hi:+.5f}]  '
                      f'd_z {v["d_z"]:+.2f}  Holm p {holm.get(other, float("nan")):.4f}')

In [ ]:
per_seed = []
for p in result_dirs:
    for j in (p / 'results').glob('NFF-MTL*__test-*.json'):
        m = json.loads(j.read_text())
        per_seed.append({'model': m['model'], 'split': m['split'], 'seed': m['seed'],
                         'ndcg@20': m['summary']['ndcg@20'], 'recall@20': m['summary']['recall@20']})
per_seed = pd.DataFrame(per_seed)
display(per_seed.pivot_table(index=['split', 'seed'], columns='model', values='ndcg@20').round(5))

## 7. Playtime head and dataset audit

The hours audit reads an existing report by default. Set RERUN_AUDIT = True in section 1 to run check_hours.py explicitly.


In [ ]:
pt = []
for p in result_dirs:
    f = p / 'playtime.json'
    if f.exists():
        m = json.loads((p / 'manifest.json').read_text())
        for name, r in json.loads(f.read_text())['metrics'].items():
            pt.append({'variant': m.get('variant'), 'seed': m.get('seed'), 'predictor': name, 'mae_log': r['mae_log'],
                       'rmse_log': r['rmse_log'], 'spearman_log': r.get('spearman_rho_log')})
pt = pd.DataFrame(pt)
display(pt.groupby(['variant', 'predictor'])[['mae_log', 'rmse_log', 'spearman_log']].agg(['mean', 'std']).round(4))

In [ ]:
CHECK, REPORT = DATA_DIR / 'check_hours.py', DATA_DIR / 'hours_report.json'
if RERUN_AUDIT:
    if not CHECK.exists():
        raise FileNotFoundError(CHECK)
    args = ['--recs', DATA_DIR / 'recommendations.csv', '--out', REPORT]
    if (DATA_DIR / 'games.csv').exists():
        args += ['--games', DATA_DIR / 'games.csv', '--meta', DATA_DIR / 'games_metadata.json']
    run(sys.executable, CHECK, *args)
if REPORT.exists():
    rep = json.loads(REPORT.read_text())
    print(json.dumps({'hours_profile': rep['hours_profile'], 'release_check': rep['release_check']}, indent=1))
else:
    print('No hours report found. Upload check_hours.py to', DATA_DIR, 'then set RERUN_AUDIT = True to run it.')

In [ ]:
OUT = DATA_DIR / 'paper_tables'
OUT.mkdir(exist_ok=True)
for s, t in tables.items():
    t.to_csv(OUT / f'comparison_{s}.csv')
per_seed.to_csv(OUT / 'nffmtl_per_seed.csv', index=False)
pt.to_csv(OUT / 'playtime_per_seed.csv', index=False)
print('written:', sorted(p.name for p in OUT.iterdir()))

## 8. Notes for any further experiment

- Run new experiments in a separate notebook and a **new** runs folder; never add them to this ledger without
  recording why. Test-year evaluations are limited by the registry next to the data cache.
- A notebook interrupt does not stop the CLI subprocess. Check with `ps -eo pid,etime,cmd | grep steam_nffmtl` in the
  terminal before restarting anything; stop a process with `kill <PID>`.
- On an L4, use `baselines.ease.device=cpu` for EASE.
- The OpenBLAS and implicit-GPU warnings in the logs affect speed only.